# Caderno de Exercícios: Análise de Sentimentos com RNN e Whisper

Bem-vindo! Este notebook contém 10 exercícios práticos desenhados para testar e expandir seu conhecimento sobre Redes Neurais Recorrentes (RNN) aplicadas à Análise de Sentimentos. 

Nos exercícios finais, você integrará seu modelo com o **OpenAI Whisper**, criando um pipeline completo que escuta um áudio, transcreve e analisa a emoção contida nele.

---
### Preparação Inicial
Certifique-se de ter as bibliotecas instaladas (descomente e rode a célula abaixo se necessário):

In [ ]:
# !pip install torch torchvision torchaudio
# !pip install openai-whisper

## Exercício 1: O problema do Padding e Vocabulário
**Objetivo:** Crie uma função chamada `build_vocab` que recebe uma lista de frases e retorna um dicionário (vocabulário). 

**Regras:**
1. O índice `0` deve ser estritamente reservado para o `<PAD>` (preenchimento).
2. O índice `1` deve ser reservado para `<UNK>` (palavras desconhecidas).
3. As palavras do dataset devem receber IDs a partir do `2`.

In [ ]:
frases_treino = ["O filme é muito bom", "Eu odiei a comida", "Foi uma experiência maravilhosa"]

def build_vocab(sentences):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    next_id = 2
    for s in sentences:
        for w in s.lower().split():
            if w not in vocab:
                vocab[w] = next_id
                next_id += 1
    return vocab

vocabulario = build_vocab(frases_treino)
print(vocabulario)

{'<PAD>': 0, '<UNK>': 1, 'o': 2, 'filme': 3, 'é': 4, 'muito': 5, 'bom': 6, 'eu': 7, 'odiei': 8, 'a': 9, 'comida': 10, 'foi': 11, 'uma': 12, 'experiência': 13, 'maravilhosa': 14}


## Exercício 2: Construindo a Vanilla RNN
**Objetivo:** Implemente a classe `SimpleRNN` herdando de `nn.Module`.

A rede deve conter:
- Uma camada `nn.Embedding`.
- Uma camada `nn.RNN` com `batch_first=True`.
- Uma camada linear `nn.Linear` que recebe o *hidden_state*.
- Uma ativação `Sigmoid` no final para retornar um valor entre 0 e 1.

In [ ]:
import torch
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, 1)

        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.embedding(x)

        out_rnn, hidden = self.rnn(x)

        out = self.fc(hidden[-1])
        return self.sigmoid(out)
    
# embedding_dim = 10
# hidden_dim = 16

# model = SimpleRNN(len(vocabulario) +1, embedding_dim, hidden_dim)

## Exercício 3: O Loop de Treinamento
**Objetivo:** Escreva um loop de treinamento clássico do PyTorch para 50 épocas. 

Você precisará:
1. Zerar os gradientes do otimizador.
2. Passar os dados (`X_train`) pelo modelo.
3. Calcular a perda usando `nn.BCELoss()`.
4. Fazer o backpropagation (`loss.backward()`).
5. Atualizar os pesos (`optimizer.step()`).

In [ ]:
# Assuma que model, X_train, y_train, criterion e optimizer já existem
epochs = 50

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.5f}")

## Exercício 4: Função de Inferência
**Objetivo:** Crie uma função `predict_sentiment(text, model, vocab, max_len)` que recebe um texto livre, converte para IDs usando o seu vocabulário, aplica o padding necessário e passa pelo modelo.
Lembre-se de usar `torch.no_grad()` e `model.eval()`.

In [ ]:
def predict_sentiment(text, model, vocab, max_len):
    model.eval()

    tokens = text.lower().split()

    unk_id = vocab.get("<UNK>", 1)
    pad_id = vocab.get("<PAD>", 0)

    seq = [vocab.get(t, unk_id) for t in tokens]

    if len(seq) < max_len:
        seq = seq + [pad_id] * (max_len - len(seq))
    else:
        seq = seq[:max_len]

    import torch
    x = torch.tensor([seq], dtype=torch.long)

    with torch.no_grad():
        out = model(x).item()

    return "Positivo" if out > 0.5 else "Negativo"

# Exemplo de uso esperado:
print(predict_sentiment("Eu achei esse produto excelente"))

## Exercício 5: Implementando Embeddings Pré-treinados (Conceitual)
**Objetivo:** Em vez de iniciar uma camada `nn.Embedding` do zero, pesquise na documentação do PyTorch como carregar matrizes de pesos estáticos. 

Crie uma instância de `nn.Embedding` usando o método `from_pretrained`. Crie um tensor mockado (falso) de tamanho (Vocab Size x 300) usando `torch.rand` apenas para simular o comportamento de importar o FastText/GloVe.

In [ ]:
import torch
import torch.nn as nn

vocab_size = len(vocabulario) + 1

mock_weights = torch.rand(vocab_size, 300)

embedding_layer = nn.Embedding.from_pretrained(mock_weights, freeze=True)

print("Embedding mock criada com shape:", embedding_layer.weight.shape)

## Exercício 6: Transcrição com Whisper
**Objetivo:** É hora de integrar ferramentas! Use a biblioteca `whisper` da OpenAI. 

Instancie o modelo na versão `'tiny'` (para rodar rápido). Crie uma função que simula o carregamento de um arquivo de áudio (você pode usar um MP3 real se tiver na mesma pasta) e retorna a string transcrita.

In [ ]:
import whisper

def transcrever_audio(caminho_arquivo):
    """Carrega o modelo 'tiny' do Whisper e transcreve o arquivo.
    Retorna a string com o texto transcrito. Em caso de erro, retorna string vazia.
    """
    try:
        model = whisper.load_model("tiny")
        result = model.transcribe(caminho_arquivo)
        return result.get("text", "")
    except Exception as e:
        print("Erro ao transcrever:", e)
        return ""

# texto = transcrever_audio("meu_audio_teste.mp3")
# print("Texto transcrito:", texto)

## Exercício 7: O Pipeline Completo (Áudio -> Texto -> Sentimento)
**Objetivo:** Junte o Exercício 4 (Sua função de previsão) com o Exercício 8 (Sua função do Whisper).

Escreva a função `audio_sentiment_pipeline(audio_path)`. Ela deve:
1. Chamar a transcrição do Whisper.
2. Imprimir o texto transcrito na tela.
3. Passar esse texto pelo seu modelo de LSTM/RNN.
4. Retornar se a pessoa falou com uma conotação Positiva ou Negativa.

In [ ]:
def audio_sentiment_pipeline(audio_path):
    texto = transcrever_audio(audio_path)

    print("Texto transcrito:", texto)

    try:
        return predict_sentiment(texto, model, vocabulario, max_len)
    except NameError:
        return "Modelo, vocab ou max_len não definidos. Transcrição: " + texto

# resultado = audio_sentiment_pipeline("reclamacao_cliente.mp3")
# print(resultado)